# URL to CSV Converter

This notebook goes through URLs of the final 50 press briefings that Gov. Jim Justice posted to his official YouTube account, and stores information from them in a CSV file, all-briefings.csv.

In [23]:
import pandas as pd
import yt_dlp
from pathlib import Path
from datetime import datetime

# Define paths
raw_folder = Path("../../data/import/raw")
clean_folder = Path("../../data/import/clean")

# Ensure clean folder exists
clean_folder.mkdir(parents=True, exist_ok=True)

# Read URLs from url.txt
url_file = raw_folder / "url.txt"
with open(url_file, 'r') as f:
    urls = [line.strip() for line in f if line.strip()]

# Get distinct URLs
distinct_urls = list(set(urls))
print(f"Found {len(urls)} total URLs, {len(distinct_urls)} distinct URLs")
for url in distinct_urls:
    print(f"  - {url}")

Found 48 total URLs, 48 distinct URLs
  - https://www.youtube.com/watch?v=2FBYWUg1B3M
  - https://www.youtube.com/watch?v=zonwVKb6kZk
  - https://www.youtube.com/watch?v=q2mZXpsBuLo
  - https://www.youtube.com/watch?v=fKaZQQaD3sw
  - https://www.youtube.com/watch?v=2I5017C_F0E
  - https://www.youtube.com/watch?v=UX8OcIXS08o
  - https://www.youtube.com/watch?v=gCELV0B8mIY
  - https://www.youtube.com/watch?v=7o8tzx9bkyA
  - https://www.youtube.com/watch?v=SZB6g4EN3xA
  - https://www.youtube.com/watch?v=-rIrKrwsPWY
  - https://www.youtube.com/watch?v=ZH-topUt2Cg
  - https://www.youtube.com/watch?v=afMfum6HwN8
  - https://www.youtube.com/watch?v=d7EGE8PywsI
  - https://www.youtube.com/watch?v=Xoq10ZFYJPI
  - https://www.youtube.com/watch?v=rWI0IrNDm_U
  - https://www.youtube.com/watch?v=c6sjnr4rTrs
  - https://www.youtube.com/watch?v=XFEzmdc8-t8
  - https://www.youtube.com/watch?v=QDGpcSbvMaw
  - https://www.youtube.com/watch?v=ui77gE-ocDw
  - https://www.youtube.com/watch?v=PTAzRJR2BVo
  

In [24]:
# Extract video metadata with retry logic
import time

video_data = []

ydl_opts = {
    'quiet': True,
    'no_warnings': True,
    'socket_timeout': 60,
}

for idx, url in enumerate(distinct_urls):
    success = False
    retries = 3
    
    while retries > 0 and not success:
        try:
            print(f"[{idx+1}/{len(distinct_urls)}] Fetching {url}...")
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(url, download=False)
                
                title = info.get('title', 'N/A')
                duration = info.get('duration', 0)
                upload_date = info.get('upload_date', 'N/A')
                
                if upload_date != 'N/A':
                    date_obj = datetime.strptime(upload_date, '%Y%m%d')
                    date_str = date_obj.strftime('%Y-%m-%d')
                else:
                    date_str = 'N/A'
                
                if duration:
                    minutes = duration // 60
                    seconds = duration % 60
                    duration_str = f"{minutes}:{seconds:02d}"
                else:
                    duration_str = 'N/A'
                
                video_data.append({
                    'URL': url,
                    'Title': title,
                    'Duration': duration_str,
                    'Date': date_str
                })
                print(f"  ✓ {title}")
                success = True
                
        except Exception as e:
            retries -= 1
            if retries > 0:
                print(f"  ⚠ Retry {4-retries}/3: {str(e)[:60]}")
                time.sleep(3)
            else:
                print(f"  ✗ Failed after retries: {str(e)[:60]}")
                video_data.append({
                    'URL': url,
                    'Title': 'Error',
                    'Duration': 'N/A',
                    'Date': 'N/A'
                })
    
    if idx < len(distinct_urls) - 1:
        time.sleep(2)

df = pd.DataFrame(video_data)
print(f"\nExtracted {len(df)} videos")
print(df)

[1/48] Fetching https://www.youtube.com/watch?v=2FBYWUg1B3M...
  ✓ Gov. Justice, First Lady Cathy Justice speak at Communities in Schools WV Conference
[2/48] Fetching https://www.youtube.com/watch?v=zonwVKb6kZk...
  ✓ Gov. Justice holds groundbreaking ceremony for section of Corridor H in Tucker County
[3/48] Fetching https://www.youtube.com/watch?v=q2mZXpsBuLo...
  ✓ Gov. Justice holds briefing on COVID-19 response - November 22, 2022
[4/48] Fetching https://www.youtube.com/watch?v=fKaZQQaD3sw...
  ✓ Gov. Justice visits the Raleigh County Commission
[5/48] Fetching https://www.youtube.com/watch?v=2I5017C_F0E...
  ✓ Gov. Justice congratulates North Central Airport for near completion of $54 million expansion
[6/48] Fetching https://www.youtube.com/watch?v=UX8OcIXS08o...
  ✓ Gov  Justice joins Congressman McKinley to make major economic development announcement in Wheeling
[7/48] Fetching https://www.youtube.com/watch?v=gCELV0B8mIY...
  ✓ Gov. Justice announces the Get Paid to Pay it F

In [22]:
# Sort by date (oldest to newest)
df['Date_sort'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.sort_values('Date_sort').drop('Date_sort', axis=1).reset_index(drop=True)

# Save to CSV
output_file = clean_folder / "all-briefings.csv"
df.to_csv(output_file, index=False)
print(f"\nSaved to: {output_file}")
print(f"Total videos: {len(df)}")


Saved to: ../../data/import/clean/all-briefings.csv
Total videos: 50
